# Evaluate best checkpoint with single-view OOF LOGO

This notebook uses the **training-time encoder definition** copied from the original training notebook, but switches evaluation to a **single-view dataset** so each file contributes its true windows once (for `window_size=500, stride=250`, typically 32 windows per file).

It evaluates one fixed checkpoint on one task with:

- encoder → file embedding (mean over windows)
- file embedding → Logistic Regression
- frog-level **OOF LOGO**
- frog probability aggregation = **mean**


In [ ]:

# =========================================================
# 0) Imports
# =========================================================
import re
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

# =========================================================
# 1) Config


# =========================================================
PROJECT_ROOT = Path("../..").resolve()
NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"
META_CSV =PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv"
CKPT_PATH = PROJECT_ROOT / "results/final/Task1717/1717_Top6_CKPT"/"seed16_step800-loss6.683.ckpt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WINDOW_SIZE = 500
STRIDE = 250
JITTER_MAX = 0

EMBED_DIM = 64
PATCH_SIZE = 50
ENCODER_STRIDE = 50
NUM_HEADS = 4

LOGREG_C = 0.01
FROG_PROB_AGG = "mean_prob"   # "mean_prob" or "median_prob"

POS_LABEL = "STP1717.1"
NEG_LABEL = "control"


# =========================================================
# 2) Exact encoder definition from the training notebook
# =========================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.register_buffer("div_term", div_term)

    def forward(self, rt):
        rt = rt.unsqueeze(-1)  # (B,seq,1)
        pe = torch.zeros(rt.size(0), rt.size(1), self.d_model, device=rt.device)
        pe[:, :, 0::2] = torch.sin(rt * self.div_term)
        pe[:, :, 1::2] = torch.cos(rt * self.div_term)
        return pe


class MassSpecWindowContrastEncoder(nn.Module):
    """
    Window-level encoder:
    Patch(Conv1d) + Transformer + CLS + RT sinusoidal positional encoding.
    Input:  signal(B,L), rt(B,L)
    Output: embedding(B,embed_dim)
    """
    def __init__(self, patch_size=50, stride=50, embed_dim=64, num_heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.embed_dim = embed_dim

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=0,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.positional_encoding = SinusoidalPositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, signal, rt):
        B = signal.size(0)

        if not torch.isfinite(signal).all():
            raise RuntimeError("signal contains NaN/Inf before encoder")
        if not torch.isfinite(rt).all():
            raise RuntimeError("rt contains NaN/Inf before encoder")

        # light numeric stabilization: keep shape, only shrink magnitude
        signal = signal / 1e6

        x = self.conv(signal.unsqueeze(1))   # (B, embed_dim, n_patches)
        if not torch.isfinite(x).all():
            raise RuntimeError("conv output contains NaN/Inf")

        x = x.permute(0, 2, 1)               # (B, n_patches, embed_dim)

        rt_patch = rt[:, self.patch_size - 1::self.stride]
        assert x.size(1) == rt_patch.size(1), (
            f"Patch/RT mismatch: conv patches={x.size(1)} vs rt patches={rt_patch.size(1)}. "
            f"patch_size={self.patch_size}, stride={self.stride}, input_len={rt.size(1)}"
        )

        rt_patch = rt_patch / 1800.0
        pe = self.positional_encoding(rt_patch)
        if not torch.isfinite(pe).all():
            raise RuntimeError("positional encoding contains NaN/Inf")

        x = x + pe
        if not torch.isfinite(x).all():
            raise RuntimeError("x + positional encoding contains NaN/Inf")

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        x = self.transformer(x)
        if not torch.isfinite(x).all():
            raise RuntimeError("transformer output contains NaN/Inf")

        h = x[:, 0, :]
        h = F.layer_norm(h, h.shape[-1:])
        if not torch.isfinite(h).all():
            raise RuntimeError("encoder output h contains NaN/Inf")

        return h





# =========================================================
# 3) Eval-only window slicer + dataset + collate
# =========================================================
class EvalWindowSlicer:
    """
    Single-view eval slicer in POINTS.
    Unlike the training dataset, this is only for inference/evaluation.
    """
    def __init__(self, window_size=500, stride=250, jitter_max=0):
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.jitter_max = int(jitter_max)
        if self.jitter_max != 0:
            raise ValueError("EvalWindowSlicer expects jitter_max=0 for deterministic evaluation")

    def __call__(self, chromatogram):
        rt = chromatogram["rt"]
        signal = chromatogram["signal"]

        if hasattr(rt, "cpu"):
            rt = rt.cpu().numpy()
        if hasattr(signal, "cpu"):
            signal = signal.cpu().numpy()

        rt = np.asarray(rt)
        signal = np.asarray(signal)

        Lsig = len(rt)
        windows = []
        start = 0

        while start + self.window_size <= Lsig:
            s = start
            e = s + self.window_size
            windows.append({
                "rt": rt[s:e].astype(np.float32),
                "signal": signal[s:e].astype(np.float32),
                "start": int(s),
            })
            start += self.stride

        return windows


class EvalWindowDataset(torch.utils.data.Dataset):
    """
    Evaluation-only dataset:
    - reads rt_grid / signal_grid
    - single view only
    - no augmentation
    - returns all real windows for one file
    """
    def __init__(self, npz_files, window_slicer):
        self.npz_files = [Path(p) for p in npz_files]
        self.slicer = window_slicer

    def __len__(self):
        return len(self.npz_files)

    def __getitem__(self, idx):
        npz_path = self.npz_files[idx]
        d = np.load(npz_path)

        chrom = {
            "rt": d["rt_grid"].astype(np.float32),
            "signal": d["signal_grid"].astype(np.float32),
            "chrom_name": npz_path.name,
        }

        wins = self.slicer(chrom)
        if len(wins) == 0:
            raise RuntimeError(f"No windows generated for file: {npz_path.name}")

        signal = np.stack([w["signal"] for w in wins], axis=0)   # (N, L)
        rt = np.stack([w["rt"] for w in wins], axis=0)           # (N, L)

        return {
            "signal": signal,
            "rt": rt,
            "file_name": npz_path.name,
        }


def collate_eval_window_level(batch):
    sigs, rts, names = [], [], []

    for b in batch:
        signal = b["signal"]   # (N, L)
        rt = b["rt"]           # (N, L)
        fname = b["file_name"]

        sigs.append(torch.from_numpy(signal))
        rts.append(torch.from_numpy(rt))
        names.extend([fname] * signal.shape[0])

    return {
        "signal": torch.cat(sigs, dim=0).float(),
        "rt": torch.cat(rts, dim=0).float(),
        "file_name": names,
    }



# =========================================================
# 4) Helpers for labels / groups / checkpoint loading
# =========================================================
def frog_from_uhm_sample(s: str) -> str:
    return str(s).split("-")[0]


def frog_from_npz_name(fn: str) -> str:
    return str(fn).split("-")[0]


def sample_id_from_npz_name(npz_name: str) -> str:
    base = Path(npz_name).name
    m = re.search(r"-(\d+)", base)
    return m.group(1) if m else base


def sample_id_from_meta_filename(meta_filename: str) -> str:
    base = str(meta_filename)
    m = re.search(r"WF22_(\d+)", base)
    return m.group(1) if m else base


def build_npz_to_label_maps(npz_files, meta_csv):
    df = pd.read_csv(meta_csv)

    assert "UHM_sample" in df.columns
    assert "treatment" in df.columns
    assert "filename" in df.columns

    meta_by_id = {}
    for _, row in df.iterrows():
        sid = sample_id_from_meta_filename(row["filename"])
        meta_by_id[str(sid)] = row

    y_str = {}
    g_frog = {}
    uhm_sample = {}
    missing = []

    for p in npz_files:
        sid = sample_id_from_npz_name(p.name)
        if str(sid) not in meta_by_id:
            missing.append(p.name)
            continue

        r = meta_by_id[str(sid)]
        uhm = str(r["UHM_sample"])
        trt = str(r["treatment"])
        frog = frog_from_uhm_sample(uhm)

        y_str[p.name] = trt
        g_frog[p.name] = frog
        uhm_sample[p.name] = uhm

    if len(missing) > 0:
        print(f"[WARN] {len(missing)} npz files not matched to metadata. Example: {missing[:5]}")

    return y_str, g_frog, uhm_sample


def strip_prefix_any(sd, prefixes):
    for p in prefixes:
        if any(k.startswith(p) for k in sd.keys()):
            out = {k[len(p):]: v for k, v in sd.items() if k.startswith(p)}
            return out, p
    return None, None


def load_encoder_from_ckpt(
    ckpt_path: str,
    device: str,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt)

    load_sd, used_prefix = strip_prefix_any(
        state,
        prefixes=["enc_with_head.encoder.", "encoder.", "model.encoder."]
    )
    if load_sd is None:
        raise KeyError(f"Cannot find encoder prefix. Example keys: {list(state.keys())[:10]}")

    enc = MassSpecWindowContrastEncoder(
        embed_dim=embed_dim,
        patch_size=patch_size,
        stride=encoder_stride,
        num_heads=num_heads,
    ).to(device)

    missing, unexpected = enc.load_state_dict(load_sd, strict=False)
    enc.eval()

    print("[CKPT LOADED]")
    print("ckpt_path   :", ckpt_path)
    print("used_prefix :", used_prefix)
    print("missing     :", len(missing), missing)
    print("unexpected  :", len(unexpected), unexpected)

    return enc



# =========================================================
# 5) Extract file embeddings
# =========================================================
@torch.no_grad()
def extract_E_file_from_encoder(encoder, loader, device, y_file_str_map):
    encoder = encoder.to(device).eval()

    Z_sum = defaultdict(lambda: None)
    Z_cnt = defaultdict(int)
    y_str = {}

    for batch in loader:
        signal = batch["signal"].to(device)   # (Bwin, L)
        rt = batch["rt"].to(device)           # (Bwin, L)
        names = list(batch["file_name"])      # len = Bwin

        z = encoder(signal, rt)               # (Bwin, D)
        z = z.detach().cpu().numpy().astype(np.float64)

        for zi, fn in zip(z, names):
            fn = str(fn)
            if fn not in y_file_str_map:
                continue

            if Z_sum[fn] is None:
                Z_sum[fn] = zi.copy()
            else:
                Z_sum[fn] += zi

            Z_cnt[fn] += 1
            y_str[fn] = y_file_str_map[fn]

    file_names = sorted(Z_sum.keys())
    E_file = np.stack([Z_sum[n] / max(1, Z_cnt[n]) for n in file_names], axis=0)
    y_file_str = np.array([y_str[n] for n in file_names], dtype=object)
    g_file = np.array([frog_from_npz_name(n) for n in file_names], dtype=str)

    # print("\n[WINDOW COUNT PER FILE]")
    # for fn in file_names[:10]:
    #     print(fn, "->", Z_cnt[fn])
    # if len(file_names) > 10:
    #     print("... total files:", len(file_names))

    return E_file, y_file_str, g_file, file_names



# =========================================================
# 6) Frog-level OOF LOGO evaluation
# =========================================================
def aggregate_by_group(values, groups, mode="mean_prob"):
    buckets = defaultdict(list)
    for v, g in zip(values, groups):
        buckets[g].append(float(v))

    group_ids = list(buckets.keys())

    if mode == "mean_prob":
        agg = np.array([np.mean(buckets[g]) for g in group_ids], dtype=float)
    elif mode == "median_prob":
        agg = np.array([np.median(buckets[g]) for g in group_ids], dtype=float)
    else:
        raise ValueError(mode)

    return agg, np.array(group_ids, dtype=object)


def logo_binary_metrics_for_task(
    E,
    y_str,
    groups_frog,
    pos_class,
    neg_class="control",
    C=10.0,
    threshold=0.5,
    frog_agg="mean_prob",
):
    E = np.asarray(E)
    y_str = np.asarray(y_str, dtype=object)
    g = np.asarray(groups_frog, dtype=object)

    keep = np.isin(y_str, [pos_class, neg_class])
    E2, y2_str, g2 = E[keep], y_str[keep], g[keep]

    if E2.shape[0] == 0:
        raise ValueError(f"No samples for {pos_class} vs {neg_class}")

    y2 = (y2_str == pos_class).astype(int)
    logo = LeaveOneGroupOut()

    P_file_all, Y_file_all, G_file_all = [], [], []
    fold_rows = []
    n_skip = 0

    for fold_idx, (tr, te) in enumerate(logo.split(E2, y2, g2), start=1):
        test_frog = np.unique(g2[te])
        assert len(test_frog) == 1
        test_frog = test_frog[0]

        if len(np.unique(y2[tr])) < 2:
            n_skip += 1
            continue

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                max_iter=5000,
                class_weight="balanced",
                solver="lbfgs",
            ),
        )
        clf.fit(E2[tr], y2[tr])

        p = clf.predict_proba(E2[te])[:, 1]
        P_file_all.append(p)
        Y_file_all.append(y2[te])
        G_file_all.append(g2[te])

        fold_rows.append({
            "fold": int(fold_idx),
            "test_frog": str(test_frog),
            "n_test_files": int(len(te)),
            "test_label": int(np.mean(y2[te]) >= 0.5),
            "prob_mean_file": float(np.mean(p)),
            "prob_median_file": float(np.median(p)),
        })

    if len(P_file_all) == 0:
        raise ValueError(f"All folds skipped. skipped={n_skip}")

    P_file = np.concatenate(P_file_all)
    Y_file = np.concatenate(Y_file_all)
    G_file = np.concatenate(G_file_all)

    P_frog, frogs = aggregate_by_group(P_file, G_file, mode=frog_agg)
    Y_frog_mean, _ = aggregate_by_group(Y_file, G_file, mode="mean_prob")
    Y_frog = (Y_frog_mean >= 0.5).astype(int)

    pred_frog = (P_frog >= threshold).astype(int)
    frog_acc = accuracy_score(Y_frog, pred_frog)

    if len(np.unique(Y_frog)) < 2:
        frog_auroc = np.nan
        frog_auprc = np.nan
    else:
        frog_auroc = roc_auc_score(Y_frog, P_frog)
        frog_auprc = average_precision_score(Y_frog, P_frog)

    frog_df = pd.DataFrame({
        "frog_id": frogs,
        "y_true": Y_frog,
        "prob_frog": P_frog,
        "pred_0.5": pred_frog,
    }).sort_values(["y_true", "prob_frog"], ascending=[False, False]).reset_index(drop=True)

    fold_df = pd.DataFrame(fold_rows)

    result = {
        "AUROC": float(frog_auroc),
        "AUPRC": float(frog_auprc),
        "ACC": float(frog_acc),
        "n_frogs": int(len(Y_frog)),
        "n_pos_frogs": int(Y_frog.sum()),
        "n_neg_frogs": int((1 - Y_frog).sum()),
        "skipped_folds": int(n_skip),
        "frog_df": frog_df,
        "fold_df": fold_df,
    }
    return result




# =========================================================
# 7) Run
# =========================================================
out_dir = Path(NPZ_DIR)
npz_files = sorted(out_dir.glob("*.npz"))
print("Total npz:", len(npz_files))

y_file_str_map, g_frog_map, uhm_sample_map = build_npz_to_label_maps(npz_files, META_CSV)

slicer = EvalWindowSlicer(
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    jitter_max=JITTER_MAX,
)

eval_ds = EvalWindowDataset(
    npz_files=npz_files,
    window_slicer=slicer,
)

eval_loader = DataLoader(
    eval_ds,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_eval_window_level,
    drop_last=False,
)

encoder = load_encoder_from_ckpt(
    CKPT_PATH,
    DEVICE,
    embed_dim=EMBED_DIM,
    patch_size=PATCH_SIZE,
    encoder_stride=ENCODER_STRIDE,
    num_heads=NUM_HEADS,
)

E_file, y_file_str, g_file, file_names = extract_E_file_from_encoder(
    encoder, eval_loader, DEVICE, y_file_str_map
)

print("\n[DATA SUMMARY]")
print("E_file shape:", E_file.shape)
print("n_files     :", len(file_names))
print("classes     :", sorted(set(y_file_str.tolist())))
print("n_frogs     :", len(set(g_file.tolist())))

res = logo_binary_metrics_for_task(
    E=E_file,
    y_str=y_file_str,
    groups_frog=g_file,
    pos_class=POS_LABEL,
    neg_class=NEG_LABEL,
    C=LOGREG_C,
    threshold=0.5,
    frog_agg=FROG_PROB_AGG,
)

print("\n[OOF LOGO frog-level result]")
print(f"ACC   : {res['ACC']:.6f}")
print(f"AUROC : {res['AUROC']:.6f}")
print(f"AUPRC : {res['AUPRC']:.6f}")
print(f"frogs : {res['n_frogs']}")

print("\n[FROG-LEVEL OOF TABLE]")
display(res["frog_df"])

print("\n[FOLD SUMMARY]")
display(res["fold_df"])


In [ ]:
from collections import defaultdict
import numpy as np
import torch

def aggregate_by_group(values, groups, mode="mean_prob"):
    buckets = defaultdict(list)
    for v, g in zip(values, groups):
        buckets[str(g)].append(float(v))
    group_ids = list(buckets.keys())
    if mode == "mean_prob":
        agg = np.array([np.mean(buckets[g]) for g in group_ids], dtype=float)
    elif mode == "median_prob":
        agg = np.array([np.median(buckets[g]) for g in group_ids], dtype=float)
    else:
        raise ValueError(mode)
    return agg, np.array(group_ids, dtype=object)

def frog_from_file_probs(p_file, g_file, method="median_prob", eps=1e-6):
    """
    method:
      - 'mean_prob'   : mean(p)
      - 'median_prob' : median(p)
     
    """
    p_file = np.asarray(p_file, dtype=float)
    g_file = np.asarray(g_file, dtype=object)

    if method == "mean_prob":
        p_frog, frogs = aggregate_by_group(p_file, g_file, mode="mean_prob")
        return p_frog, frogs

    if method == "median_prob":
        p_frog, frogs = aggregate_by_group(p_file, g_file, mode="median_prob")
        return p_frog, frogs

    # if method == "mean_logit":
    #     p_clip = np.clip(p_file, eps, 1 - eps)
    #     l = logit(p_clip)
    #     l_frog, frogs = aggregate_by_group(l, g_file, mode="mean")
    #     return expit(l_frog), frogs

    raise ValueError(f"Unknown method: {method}")


    
def logo_binary_metrics_file_and_frog_probs(
    E, y_str, groups_frog, pos_class, neg_class,
    C=10.0, threshold=0.5,
    frog_agg="mean_prob",  # or "mean_logit"
):
    E = np.asarray(E)
    y_str = np.asarray(y_str, dtype=object)
    g = np.asarray(groups_frog, dtype=object)

    keep = np.isin(y_str, [pos_class, neg_class])
    E2, y2_str, g2 = E[keep], y_str[keep], g[keep]
    if E2.shape[0] == 0:
        raise ValueError(f"No samples for {pos_class} vs {neg_class}")
    y2 = (y2_str == pos_class).astype(int)

    logo = LeaveOneGroupOut()

    P_file_all, Y_file_all, G_file_all = [], [], []
    n_skip = 0

    for tr, te in logo.split(E2, y2, g2):
        if len(np.unique(y2[tr])) < 2:
            n_skip += 1
            continue

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=C, max_iter=5000, class_weight="balanced", solver="lbfgs"),
        )
        clf.fit(E2[tr], y2[tr])

        p = clf.predict_proba(E2[te])[:, 1]
        P_file_all.append(p)
        Y_file_all.append(y2[te])
        G_file_all.append(g2[te])

    if len(P_file_all) == 0:
        raise ValueError(f"All folds skipped because TRAIN lacks 2 classes. Skipped={n_skip}")

    P_file = np.concatenate(P_file_all)
    Y_file = np.concatenate(Y_file_all)
    G_file = np.concatenate(G_file_all)

    # ---- file-level metrics (OOF) ----
    pred_file = (P_file >= threshold).astype(int)
    file_acc = accuracy_score(Y_file, pred_file)
    if len(np.unique(Y_file)) < 2:
        file_auroc = np.nan
        file_auprc = np.nan
    else:
        file_auroc = roc_auc_score(Y_file, P_file)
        file_auprc = average_precision_score(Y_file, P_file)

    # ---- frog-level metrics: aggregate OOF probs within each frog ----
    P_frog, frogs = frog_from_file_probs(P_file, G_file, method=frog_agg)
    # frog label：同一 frog 应该同 label（否则数据有问题），这里用 majority/first 都行
    # 我用“该frog的 file labels 的平均>=0.5”
    Y_frog_mean, _ = aggregate_by_group(Y_file, G_file, mode="mean_prob")
    Y_frog = (Y_frog_mean >= 0.5).astype(int)

    pred_frog = (P_frog >= threshold).astype(int)
    frog_acc = accuracy_score(Y_frog, pred_frog)
    if len(np.unique(Y_frog)) < 2:
        frog_auroc = np.nan
        frog_auprc = np.nan
    else:
        frog_auroc = roc_auc_score(Y_frog, P_frog)
        frog_auprc = average_precision_score(Y_frog, P_frog)

    return {
        #"file_level": {"AUROC": float(file_auroc), "AUPRC": float(file_auprc), "ACC": float(file_acc), "n": int(len(Y_file)), "pos": int(Y_file.sum()), "neg": int((1-Y_file).sum())},
        "frog_level": {"AUROC": float(frog_auroc), "AUPRC": float(frog_auprc), "ACC": float(frog_acc),
                       "n": int(len(Y_frog)), "pos": int(Y_frog.sum()), "neg": int((1-Y_frog).sum())},
        #"skipped_folds": int(n_skip),
        "frog_agg": frog_agg,
    }



def _overlap_any_interval(rt_start, rt_end, intervals):
    """
    rt_start, rt_end: torch.Tensor shape (B,)
    intervals: list of (low, high) in seconds
    return: torch.BoolTensor (B,) True if overlaps any interval
    """
    if intervals is None or len(intervals) == 0:
        return torch.zeros_like(rt_start, dtype=torch.bool)

    hit = torch.zeros_like(rt_start, dtype=torch.bool)
    for lo, hi in intervals:
        lo = float(lo); hi = float(hi)
        hit = hit | ((rt_end >= lo) & (rt_start <= hi))
    return hit

@torch.no_grad()
def extract_E_file_from_encoder(
    encoder, loader, device,
    mode="all",              # "all" | "mask" | "only"
    intervals=None,          # list[(lo,hi)] in seconds
    min_windows_per_file=1,  # if < this, skip that file
    verbose=True,
):
    """
    windows -> (optional RT filter) -> encoder -> mean pooling per file => E_file

    mode:
      - "all" : use all windows
      - "mask": drop windows overlapping intervals
      - "only": keep only windows overlapping intervals
    """
    encoder = encoder.to(device).eval()

    Z_sum = defaultdict(lambda: None)
    Z_cnt = defaultdict(int)
    y_str = {}

    total_windows = 0
    kept_windows = 0

    for batch in loader:
        signal = batch["signal"].to(device)   # (Bwin, L)
        rt     = batch["rt"].to(device)       # (Bwin, L)
        names  = list(batch["file_name"])     # list[str], len=Bwin

        B = rt.shape[0]
        total_windows += B

        if mode == "all":
            keep = torch.ones(B, dtype=torch.bool, device=device)
        else:
            rt_start = rt[:, 0]
            rt_end   = rt[:, -1]
            overlap = _overlap_any_interval(rt_start, rt_end, intervals)
            if mode == "mask":
                keep = ~overlap
            elif mode == "only":
                keep = overlap
            else:
                raise ValueError("mode must be all/mask/only")

        k = int(keep.sum().item())
        kept_windows += k
        if k == 0:
            continue

        # filter
        signal_k = signal[keep]
        rt_k     = rt[keep]
        names_k  = [n for n, kk in zip(names, keep.detach().cpu().numpy().tolist()) if kk]

        # encode
        z = encoder(signal_k, rt_k)  # (k, D)
        z = z.detach().cpu().numpy().astype(np.float64)

        # mean pooling per file
        for zi, fn in zip(z, names_k):
            fn = str(fn)
            if fn not in y_file_str_map:
                continue
            if Z_sum[fn] is None:
                Z_sum[fn] = zi.copy()
            else:
                Z_sum[fn] += zi
            Z_cnt[fn] += 1
            y_str[fn] = y_file_str_map[fn]

    # finalize files
    file_names_all = sorted(Z_sum.keys())
    file_names = [fn for fn in file_names_all if Z_cnt[fn] >= min_windows_per_file]

    dropped_files = len(file_names_all) - len(file_names)
    if verbose:
        print(f"[extract] mode={mode} intervals={intervals} kept_windows={kept_windows}/{total_windows} "
              f"files={len(file_names)} (dropped_files={dropped_files}, min_win={min_windows_per_file})")

    if len(file_names) == 0:
        raise ValueError("No files left after RT filtering. Try larger intervals or lower min_windows_per_file.")

    E_file = np.stack([Z_sum[n] / Z_cnt[n] for n in file_names], axis=0)
    y_file_str = np.array([y_str[n] for n in file_names], dtype=object)
    g_file = np.array([str(n).split("-")[0] for n in file_names], dtype=str)  # UHMxx-... -> UHMxx

    return E_file, y_file_str, g_file, file_names

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import torch
import matplotlib.pyplot as plt

# =========================================================
# 1) Cache RAW windows once (not embeddings)
# =========================================================
@torch.no_grad()
def build_raw_window_cache(loader):
    """
    Returns
    -------
    signal_all : (Nwin, L) float32
    rt_all     : (Nwin, L) float32
    fn_list    : list[str], len Nwin
    """
    signal_list = []
    rt_list = []
    fn_list = []

    for batch in loader:
        signal = batch["signal"].cpu().numpy().astype(np.float32)   # (Bwin, L)
        rt     = batch["rt"].cpu().numpy().astype(np.float32)       # (Bwin, L)
        names  = list(batch["file_name"])                           # len = Bwin

        signal_list.append(signal)
        rt_list.append(rt)
        fn_list.extend([str(n) for n in names])

    signal_all = np.concatenate(signal_list, axis=0)
    rt_all = np.concatenate(rt_list, axis=0)

    return signal_all, rt_all, fn_list


# =========================================================
# 2) RT-bin masking on RAW signal
# =========================================================
def apply_raw_rt_mask(signal_all, rt_all, lo, hi, mode="mask", fill_value=0.0):
    """
    signal_all : (Nwin, L)
    rt_all     : (Nwin, L)
    lo, hi     : float, RT bin in seconds

    mode:
      - "mask": zero-out ONLY the region inside [lo, hi)
      - "only": keep ONLY the region inside [lo, hi), zero outside

    Returns
    -------
    signal_masked : (Nwin, L) float32
    """
    sig = signal_all.copy()
    keep = (rt_all >= float(lo)) & (rt_all < float(hi))

    if mode == "mask":
        sig[keep] = fill_value
    elif mode == "only":
        sig[~keep] = fill_value
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return sig.astype(np.float32)


# =========================================================
# 3) Re-encode RAW windows in batches
# =========================================================
@torch.no_grad()
def encode_windows_from_raw_cache(
    encoder,
    signal_all,
    rt_all,
    device,
    batch_size=512,
    normalize=False,
):
    """
    Returns
    -------
    Z_all : (Nwin, D) float32
    """
    encoder = encoder.to(device).eval()
    Z_list = []

    n = signal_all.shape[0]
    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)

        sig = torch.from_numpy(signal_all[s:e]).to(device).float()
        rt  = torch.from_numpy(rt_all[s:e]).to(device).float()

        z = encoder(sig, rt)
        if normalize:
            z = torch.nn.functional.normalize(z, dim=-1)

        Z_list.append(z.detach().cpu().numpy().astype(np.float32))

    return np.concatenate(Z_list, axis=0)


# =========================================================
# 4) Mean pool windows -> file embeddings
# =========================================================
def mean_pool_to_file_embeddings(
    Z_all,
    fn_list,
    y_file_str_map,
):
    Z_sum = defaultdict(lambda: None)
    Z_cnt = defaultdict(int)
    y_str = {}

    for z, fn in zip(Z_all, fn_list):
        if fn not in y_file_str_map:
            continue

        if Z_sum[fn] is None:
            Z_sum[fn] = z.astype(np.float64).copy()
        else:
            Z_sum[fn] += z.astype(np.float64)

        Z_cnt[fn] += 1
        y_str[fn] = y_file_str_map[fn]

    file_names = sorted(Z_sum.keys())
    if len(file_names) == 0:
        raise ValueError("No files left after pooling")

    E_file = np.stack(
        [(Z_sum[fn] / max(1, Z_cnt[fn])).astype(np.float32) for fn in file_names],
        axis=0
    )
    y_file_str = np.array([y_str[fn] for fn in file_names], dtype=object)
    g_file = np.array([fn.split("-")[0] for fn in file_names], dtype=str)

    stats = {
        "n_files": int(len(file_names)),
        "n_frogs": int(len(set(g_file))),
        "windows_per_file_min": int(min(Z_cnt[fn] for fn in file_names)),
        "windows_per_file_median": float(np.median([Z_cnt[fn] for fn in file_names])),
        "windows_per_file_max": int(max(Z_cnt[fn] for fn in file_names)),
    }
    return E_file, y_file_str, g_file, file_names, stats


# =========================================================
# 5) Raw-signal ablation curve
# =========================================================
def delta_auc_curve_rawsignal(
    encoder,
    signal_all,
    rt_all,
    fn_list,
    y_file_str_map,
    pos_label,
    neg_label,
    bin_size=100.0,
    C=40.0,
    frog_agg="mean_prob",
    rt_min=None,
    rt_max=None,
    align_bins=True,
    mode="mask",          # "mask" or "only"
    fill_value=0.0,
    batch_size=512,
    normalize=False,
    verbose=True,
):
    """
    True raw-signal ablation:
      for each RT bin:
        1) modify raw signal values
        2) re-encode windows
        3) pool to file
        4) LOGO frog-level evaluation
    """
    signal_all = np.asarray(signal_all, dtype=np.float32)
    rt_all = np.asarray(rt_all, dtype=np.float32)

    rt_s = rt_all[:, 0]
    rt_e = rt_all[:, -1]

    if rt_min is None:
        rt_min = float(np.min(rt_s))
    if rt_max is None:
        rt_max = float(np.max(rt_e))

    if align_bins:
        rt0 = np.floor(rt_min / bin_size) * bin_size
        rt_min_use = float(rt0)
    else:
        rt_min_use = float(rt_min)

    edges = np.arange(rt_min_use, rt_max + bin_size, bin_size, dtype=float)
    bins = [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]

    # -------------------------
    # baseline: no masking
    # -------------------------
    Z0 = encode_windows_from_raw_cache(
        encoder=encoder,
        signal_all=signal_all,
        rt_all=rt_all,
        device=DEVICE,
        batch_size=batch_size,
        normalize=normalize,
    )

    E0, y0, g0, f0, st0 = mean_pool_to_file_embeddings(
        Z0, fn_list, y_file_str_map
    )

    r0 = logo_binary_metrics_file_and_frog_probs(
        E0, y0, g0,
        pos_class=pos_label,
        neg_class=neg_label,
        C=C,
        threshold=0.5,
        frog_agg=frog_agg,
    )
    auc_base = r0["frog_level"]["AUROC"]

    if verbose:
        print(f"[baseline-raw-{mode}] {pos_label} vs {neg_label} | "
              f"AUC={auc_base:.4f} | files={len(f0)} frogs={len(set(g0))}")

    rows = []

    for lo, hi in bins:
        sig_mod = apply_raw_rt_mask(
            signal_all=signal_all,
            rt_all=rt_all,
            lo=lo,
            hi=hi,
            mode=mode,
            fill_value=fill_value,
        )

        try:
            Zm = encode_windows_from_raw_cache(
                encoder=encoder,
                signal_all=sig_mod,
                rt_all=rt_all,
                device=DEVICE,
                batch_size=batch_size,
                normalize=normalize,
            )

            E, y, g, f, st = mean_pool_to_file_embeddings(
                Zm, fn_list, y_file_str_map
            )

            rr = logo_binary_metrics_file_and_frog_probs(
                E, y, g,
                pos_class=pos_label,
                neg_class=neg_label,
                C=C,
                threshold=0.5,
                frog_agg=frog_agg,
            )

            auc_mod = rr["frog_level"]["AUROC"]

            if mode == "mask":
                delta = auc_base - auc_mod
            elif mode == "only":
                delta = auc_mod - auc_base
            else:
                raise ValueError(mode)

            rows.append({
                "mode": mode,
                "rt_lo": lo,
                "rt_hi": hi,
                "rt_mid": 0.5 * (lo + hi),
                "AUC_base": auc_base,
                "AUC_ablation": auc_mod,
                "DELTA": delta,
                "n_files": st["n_files"],
                "n_frogs": st["n_frogs"],
                "windows_per_file_min": st["windows_per_file_min"],
                "windows_per_file_median": st["windows_per_file_median"],
                "windows_per_file_max": st["windows_per_file_max"],
            })

        except Exception as e:
            rows.append({
                "mode": mode,
                "rt_lo": lo,
                "rt_hi": hi,
                "rt_mid": 0.5 * (lo + hi),
                "AUC_base": auc_base,
                "AUC_ablation": np.nan,
                "DELTA": np.nan,
                "n_files": 0,
                "n_frogs": 0,
                "windows_per_file_min": np.nan,
                "windows_per_file_median": np.nan,
                "windows_per_file_max": np.nan,
                "err": repr(e),
            })

    return pd.DataFrame(rows)


# =========================================================
# 6) Plot helper
# =========================================================
def plot_delta_curve_raw(df, title):
    dfp = df.dropna(subset=["DELTA"]).sort_values("rt_mid")

    plt.figure(figsize=(10, 4))
    plt.plot(dfp["rt_mid"].values, dfp["DELTA"].values, marker="o", linewidth=1)
    plt.axhline(0.0, linewidth=1)
    plt.xlabel("RT bin midpoint (s)")

    mode = str(dfp["mode"].iloc[0]) if len(dfp) > 0 else "mask"
    if mode == "mask":
        plt.ylabel("ΔAUC = AUC_base - AUC_mask(bin)")
    else:
        plt.ylabel("ΔAUC = AUC_only(bin) - AUC_base")

    plt.title(title)
    plt.tight_layout()
    plt.show()


# =========================================================
# 7) RUN
# =========================================================
signal_all, rt_all, fn_list = build_raw_window_cache(eval_loader)

print("[raw-cache] windows:", signal_all.shape[0], "L:", signal_all.shape[1],
      "rt_min/max:", float(rt_all[:, 0].min()), float(rt_all[:, -1].max()))

cnt = Counter(fn_list)
vals = np.array(list(cnt.values()))
print("[raw-cache] windows per file: min/median/max =",
      int(vals.min()), float(np.median(vals)), int(vals.max()))




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1) 输入

# ============================================================
#PROJECT_ROOT = Path("../..").resolve()
DIR_BIN100 = PROJECT_ROOT / "results/final/Task1717/1717_Ablation" / "1717_ablation_multi_ckpt_outputs_top6seeds_bin100"
DIR_BIN50  = PROJECT_ROOT / "results/final/Task1717/1717_Ablation" / "1717_ablation_multi_ckpt_outputs_top6seeds_bin50"

CSV_MASK_100 = DIR_BIN100 / "ALL_mask__bin100__STP1717.1_vs_control.csv"
CSV_MASK_50  = DIR_BIN50  / "ALL_mask__bin50__STP1717.1_vs_control.csv"
CSV_ONLY_50  = DIR_BIN50  / "ALL_only_bin50_STP1717.1_vs_control.csv"

df_mask_100 = pd.read_csv(CSV_MASK_100)
df_mask_50  = pd.read_csv(CSV_MASK_50)
df_only_50  = pd.read_csv(CSV_ONLY_50)
df = df_mask_50.copy()
x_col = "rt_mid"
y_col = "DELTA"

# ============================================================
# 2) Mean ΔAUC curve across top K ckpts
# ============================================================
df_mean = (
    df.groupby(x_col, as_index=False)[y_col]
      .mean()
      .sort_values(x_col)
      .reset_index(drop=True)
)

x = df_mean[x_col].values
y = df_mean[y_col].values

# ============================================================
# 3) 找局部峰
#    只保留明显正峰
# ============================================================
min_peak = 0.05   # 你可以试 0.04 / 0.05 / 0.06

peak_idx = []
for i in range(1, len(y) - 1):
    if y[i] > y[i-1] and y[i] > y[i+1] and y[i] >= min_peak:
        peak_idx.append(i)

# ============================================================
# 3) Select fixed 50s top bins
#    不找 peak，不向左右扩展，因此不会出现重叠区间
# ============================================================
TOP_K = 10
BIN_SIZE = 50

# 这个参数用于避免选到相邻 bin
# 100 表示两个 hotspot center 至少间隔 100s
# 如果你想允许 1100–1150 和 1150–1200 这种相邻 bin 都入选，改成 50
MIN_CENTER_DISTANCE = 50

# 可以设成 0.05 或 0.08，过滤太弱的 bin
MIN_DELTA = 0.01

df_candidates = (
    df_mean[df_mean[y_col] >= MIN_DELTA]
    .sort_values(y_col, ascending=False)
    .reset_index(drop=True)
)

regions = []

for _, row in df_candidates.iterrows():
    center = float(row[x_col])
    delta = float(row[y_col])

    keep = True
    for reg in regions:
        if abs(center - reg["peak_x"]) < MIN_CENTER_DISTANCE:
            keep = False
            break

    if keep:
        regions.append({
            "start": center - BIN_SIZE / 2,
            "end": center + BIN_SIZE / 2,
            "peak_x": center,
            "peak_val": delta,
            "center": center,
            "delta": delta,
        })

    if len(regions) >= TOP_K:
        break


# ============================================================
# 6) 打印排序结果
# ============================================================
print("Ranked fixed-bin hotspot regions:")
for i, reg in enumerate(regions, start=1):
    print(
        f"R{i}: {reg['start']:.0f}–{reg['end']:.0f} s | "
        f"center {reg['center']:.0f} s | mean ΔAUC={reg['delta']:.3f}"
    )

# Optional: save selected regions
df_regions = pd.DataFrame([
    {
        "region": f"R{i}",
        "rt_lo": reg["start"],
        "rt_hi": reg["end"],
        "rt_mid": reg["center"],
        "mean_DELTA": reg["delta"],
    }
    for i, reg in enumerate(regions, start=1)
])

# OUT_DIR = DIR_BIN50 / "fixed_topbin_hotspots"
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# df_regions.to_csv(OUT_DIR / "selected_fixed_50s_topbins.csv", index=False)

display(df_regions)



In [ ]:
import os
import re
from pathlib import Path
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ============================================================
# 0) 先取前3个 hotspot
#    你前面已经有 regions（按 peak_val 排好序）
# ============================================================
top_regions = regions[:3]

print("Top 3 hotspot regions:")
for i, reg in enumerate(top_regions, start=1):
    print(
        f"R{i}: {reg['start']:.0f}–{reg['end']:.0f} s | "
        f"peak at {reg['peak_x']:.0f} s | peak ΔAUC={reg['peak_val']:.3f}"
    )

# ============================================================
# 1) 生成所有组合
# ============================================================
combo_defs = []

# 单个
combo_defs.append(("R1", [0]))
combo_defs.append(("R2", [1]))
combo_defs.append(("R3", [2]))

# 两两
combo_defs.append(("R1+R2", [0, 1]))
combo_defs.append(("R1+R3", [0, 2]))
combo_defs.append(("R2+R3", [1, 2]))

# 三个
combo_defs.append(("R1+R2+R3", [0, 1, 2]))

print("\nCombinations to evaluate:")
for name, idxs in combo_defs:
    print(name, idxs)

# ============================================================
# 2) top3 ckpt paths
#    从 df_mask_50 里取 ckpt_name，再去 BestCKPT_top3 匹配
# ============================================================
# CKPT_DIR = Path("./1717_BestCKPT_top3")
CKPT_DIR =PROJECT_ROOT / "results/final/Task1717/1717_Top6_CKPT"

def resolve_ckpt_path(ckpt_name, ckpt_dir=CKPT_DIR):
    ckpt_name = str(ckpt_name)
    cands = list(ckpt_dir.glob("*.ckpt"))

    # exact
    for p in cands:
        if p.name == ckpt_name:
            return str(p)

    # stem contains
    stem = ckpt_name.replace(".ckpt", "")
    for p in cands:
        if stem in p.name:
            return str(p)

    # reverse contains
    for p in cands:
        if p.name.replace(".ckpt", "") in stem:
            return str(p)

    raise FileNotFoundError(f"Cannot resolve ckpt path for: {ckpt_name} in {ckpt_dir}")

top3_ckpt_names = sorted(df_mask_50["ckpt_name"].unique().tolist())
top3_ckpt_paths = [resolve_ckpt_path(n) for n in top3_ckpt_names]

print("\nTop3 checkpoints:")
for p in top3_ckpt_paths:
    print(" -", p)

# ============================================================
# 3) helper: frog id from filename
# ============================================================
def frog_from_npz_name_local(fn):
    m = re.search(r"(UHM\d+)", str(fn))
    if m:
        return m.group(1)
    s = os.path.basename(str(fn))
    return re.split(r"[_\-]", s)[0]

# ============================================================
# 4) helper: apply regions to raw signal
# ============================================================
def apply_regions_to_signal(signal_all, rt_all, region_list, mode="mask", fill_value=0.0):
    """
    signal_all: (N_windows, L)
    rt_all    : (N_windows, L)
    region_list: [(start, end), ...]
    mode: "mask" or "only"
    """
    signal_mod = signal_all.copy()

    keep_mask = np.zeros_like(signal_mod, dtype=bool)
    for (start, end) in region_list:
        keep_mask |= (rt_all >= start) & (rt_all <= end)

    if mode == "mask":
        signal_mod[keep_mask] = fill_value
    elif mode == "only":
        signal_mod[~keep_mask] = fill_value
    else:
        raise ValueError(f"Unknown mode={mode}")

    return signal_mod

# ============================================================
# 5) helper: encode modified raw windows
# ============================================================
@torch.no_grad()
def encode_windows_with_encoder(encoder, signal_all_mod, rt_all, batch_size=512, device="cuda"):
    encoder.eval()
    out = []

    n = signal_all_mod.shape[0]
    for i in range(0, n, batch_size):
        s = torch.tensor(signal_all_mod[i:i+batch_size], dtype=torch.float32, device=device)
        r = torch.tensor(rt_all[i:i+batch_size], dtype=torch.float32, device=device)

        z = encoder(s, r)
        if isinstance(z, (tuple, list)):
            z = z[0]

        z = z.detach().cpu().numpy()
        out.append(z)

    return np.concatenate(out, axis=0)

# ============================================================
# 6) helper: window embedding -> file embedding -> frog LOGO AUROC
# ============================================================
def logo_frog_auroc_from_window_embeddings(
    E_windows,
    fn_list,
    y_file_str_map,
    pos_label="STP1717.1",
    neg_label="control",
    C=10.0,
    frog_agg="mean_prob",
):
    # aggregate windows -> file embedding
    file_to_embs = defaultdict(list)
    for e, fn in zip(E_windows, fn_list):
        file_to_embs[fn].append(e)

    files = []
    X_file = []
    y_file = []
    frog_file = []

    for fn, embs in file_to_embs.items():
        if fn not in y_file_str_map:
            continue
        lab = y_file_str_map[fn]
        if lab not in [pos_label, neg_label]:
            continue

        files.append(fn)
        X_file.append(np.mean(np.stack(embs, axis=0), axis=0))
        y_file.append(1 if lab == pos_label else 0)
        frog_file.append(frog_from_npz_name_local(fn))

    X_file = np.asarray(X_file)
    y_file = np.asarray(y_file)
    frog_file = np.asarray(frog_file)

    uniq_frogs = np.unique(frog_file)
    frog_probs = []
    frog_true = []

    for test_frog in uniq_frogs:
        te = frog_file == test_frog
        tr = ~te

        Xtr, Xte = X_file[tr], X_file[te]
        ytr, yte = y_file[tr], y_file[te]

        if len(np.unique(ytr)) < 2:
            continue

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)

        clf = LogisticRegression(
            C=C,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=5000,
            random_state=0,
        )
        clf.fit(Xtr_s, ytr)
        p_file = clf.predict_proba(Xte_s)[:, 1]

        if frog_agg == "mean_prob":
            p_frog = float(np.mean(p_file))
        elif frog_agg == "median_prob":
            p_frog = float(np.median(p_file))
        else:
            raise ValueError(f"Unknown frog_agg={frog_agg}")

        frog_probs.append(p_frog)
        frog_true.append(int(np.round(np.mean(yte))))

    frog_probs = np.asarray(frog_probs)
    frog_true = np.asarray(frog_true)

    if len(np.unique(frog_true)) < 2:
        return np.nan

    return roc_auc_score(frog_true, frog_probs)

# ============================================================
# 7) helper: evaluate one ckpt under a combination
# ============================================================
def eval_one_ckpt_combo(
    ckpt_path,
    region_list,
    signal_all,
    rt_all,
    fn_list,
    y_file_str_map,
    mode="mask",
    fill_value=0.0,
    batch_size=512,
    pos_label="STP1717.1",
    neg_label="control",
):
    encoder = load_encoder_from_ckpt(
        ckpt_path,
        DEVICE,
        embed_dim=EMBED_DIM,
        patch_size=PATCH_SIZE,
        encoder_stride=ENCODER_STRIDE,
        num_heads=NUM_HEADS,
    )

    # baseline
    E_base = encode_windows_with_encoder(
        encoder=encoder,
        signal_all_mod=signal_all,
        rt_all=rt_all,
        batch_size=batch_size,
        device=DEVICE,
    )
    auc_base = logo_frog_auroc_from_window_embeddings(
        E_windows=E_base,
        fn_list=fn_list,
        y_file_str_map=y_file_str_map,
        pos_label=pos_label,
        neg_label=neg_label,
        C=LOGREG_C,
        frog_agg=FROG_PROB_AGG,
    )

    # modified
    signal_mod = apply_regions_to_signal(
        signal_all=signal_all,
        rt_all=rt_all,
        region_list=region_list,
        mode=mode,
        fill_value=fill_value,
    )

    E_mod = encode_windows_with_encoder(
        encoder=encoder,
        signal_all_mod=signal_mod,
        rt_all=rt_all,
        batch_size=batch_size,
        device=DEVICE,
    )
    auc_mod = logo_frog_auroc_from_window_embeddings(
        E_windows=E_mod,
        fn_list=fn_list,
        y_file_str_map=y_file_str_map,
        pos_label=pos_label,
        neg_label=neg_label,
        C=LOGREG_C,
        frog_agg=FROG_PROB_AGG,
    )

    return auc_base, auc_mod

# ============================================================
# 8) 跑 full combinatorial evaluation
# ============================================================
rows = []

for ckpt_name, ckpt_path in zip(top3_ckpt_names, top3_ckpt_paths):
    print(f"\n[CKPT] {ckpt_name}")

    for combo_label, idxs in combo_defs:
        regset = [(top_regions[i]["start"], top_regions[i]["end"]) for i in idxs]
        print(f"  combo: {combo_label} -> {regset}")

        # MASK
        auc_base_m, auc_masked = eval_one_ckpt_combo(
            ckpt_path=ckpt_path,
            region_list=regset,
            signal_all=signal_all,
            rt_all=rt_all,
            fn_list=fn_list,
            y_file_str_map=y_file_str_map,
            mode="mask",
            fill_value=0.0,
            batch_size=512,
            pos_label="STP1717.1",
            neg_label="control",
        )

        # ONLY
        auc_base_o, auc_only = eval_one_ckpt_combo(
            ckpt_path=ckpt_path,
            region_list=regset,
            signal_all=signal_all,
            rt_all=rt_all,
            fn_list=fn_list,
            y_file_str_map=y_file_str_map,
            mode="only",
            fill_value=0.0,
            batch_size=512,
            pos_label="STP1717.1",
            neg_label="control",
        )

        rows.append({
            "ckpt_name": ckpt_name,
            "combo_label": combo_label,
            "region_list": str(regset),
            "AUC_base_mask": auc_base_m,
            "AUC_masked": auc_masked,
            "DELTA_mask": auc_base_m - auc_masked,
            "AUC_base_only": auc_base_o,
            "AUC_only": auc_only,
        })

df_combo = pd.DataFrame(rows)

print("\nRaw combinatorial results:")
display(df_combo)

# ============================================================
# 9) mean across top3
# ============================================================
df_combo_mean = (
    df_combo.groupby("combo_label", as_index=False)[
        ["AUC_base_mask", "AUC_masked", "DELTA_mask", "AUC_base_only", "AUC_only"]
    ]
    .mean()
)

# 固定顺序
combo_order = ["R1", "R2", "R3", "R1+R2", "R1+R3", "R2+R3", "R1+R2+R3"]
df_combo_mean["combo_label"] = pd.Categorical(
    df_combo_mean["combo_label"],
    categories=combo_order,
    ordered=True
)
df_combo_mean = df_combo_mean.sort_values("combo_label").reset_index(drop=True)

print("\nMean across top3:")
display(df_combo_mean)

# ============================================================
# 10) save (optional)
# ============================================================

# OUT_DIR = Path("./hotspot_combinatorial_outputs")
# OUT_DIR.mkdir(exist_ok=True, parents=True)

# CKPT_PATH = PROJECT_ROOT / "results/final/Task1717/1717_Top6_CKPT"/"seed5_step400-loss6.692.ckpt"
OUT_DIR =  PROJECT_ROOT / "results/final/Task1717/1717_Ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_mask_50.to_csv(OUT_DIR / "df_mask_50_all_ckpt.csv", index=False)
df_mean.to_csv(OUT_DIR / "df_mean_delta_curve.csv", index=False)
df_regions = pd.DataFrame([
    {
        "region": f"R{i}",
        "rt_lo": reg["start"],
        "rt_hi": reg["end"],
        "rt_mid": reg["center"],
        "mean_DELTA": reg["delta"],
    }
    for i, reg in enumerate(regions, start=1)
])

df_regions.to_csv(OUT_DIR / "df_selected_regions.csv", index=False)

df_combo_mean.to_csv(OUT_DIR / "df_combo_mean.csv", index=False)
df_combo.to_csv(OUT_DIR / "df_combo_raw.csv", index=False)



print("Saved:", OUT_DIR / "1717_top6_df_combo_raw.csv")
print("Saved:", OUT_DIR / "1717_top6_df_combo_mean.csv")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 你已有数据
# ============================================================
# df_mask_50, df_mean, regions, df_combo_mean

# ============================================================
# 组合标签（顺序固定）
# ============================================================
order = ["R1", "R2", "R3", "R1+R2", "R1+R3", "R2+R3", "R1+R2+R3"]

df_plot = df_combo_mean.set_index("combo_label").loc[order].reset_index()

# ============================================================
# baseline
# ============================================================
baseline_auc = df_plot["AUC_base_mask"].iloc[0]

# ============================================================
# 画图
# ============================================================
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = fig.add_gridspec(2, 2, height_ratios=[2.2, 1.4], hspace=0.35, wspace=0.3)

MAIN_COLOR = "#2A6DB0"     # 深蓝（主）
LIGHT_COLOR = "#9BBCE0"    # 浅蓝（细线）
SHADE_COLOR = "#D6DFEA"    # 区间阴影
BASELINE_COLOR = "#2A6DB0" # baseline统一
# ============================================================
# A. RT hotspot 曲线（上面整行）
# ============================================================
ax0 = fig.add_subplot(gs[0, :])

# 所有 ckpt（淡）
for _, g in df_mask_50.groupby("ckpt_name"):
    g = g.sort_values("rt_mid")
    ax0.plot(g["rt_mid"], g["DELTA"], color=LIGHT_COLOR, alpha=0.2, lw=1)

# mean
ax0.plot(df_mean["rt_mid"], df_mean["DELTA"], color=MAIN_COLOR, lw=3)

# hotspot 阴影
for i, reg in enumerate(regions[:9], start=1):
    ax0.axvspan(reg["start"], reg["end"], color=SHADE_COLOR, alpha=0.3)

    mid = (reg["start"] + reg["end"]) / 2
    ymax = max(df_mean["DELTA"].max(), 0.01)
    ax0.text(mid, ymax * 1.05, f"R{i}", ha="center", fontsize=10)

ax0.axhline(0, color="#777777", lw=0.8)

ax0.set_title(
    "Identification of discriminative RT regions via ΔAUC ablation (top-3 averaged model)",
    fontsize=12,
)
ax0.set_xlabel("RT (s)")
ax0.set_ylabel("ΔAUC")

ax0.spines["top"].set_visible(False)
ax0.spines["right"].set_visible(False)

ax0.text(
    0.01, 0.98,
    "Task: STP1717.1 vs Control",
    transform=ax0.transAxes,
    fontsize=9,
    verticalalignment="top"
)

# ============================================================
# B. Mask importance（左下）
# ============================================================
ax1 = fig.add_subplot(gs[1, 0])

ax1.bar(df_plot["combo_label"], df_plot["DELTA_mask"])

ax1.axhline(0, color=MAIN_COLOR, lw=0.8)

ax1.text(
    0.02,
    0.92,
    f"Baseline AUROC: {baseline_auc:.3f}",
    transform=ax1.transAxes,
    fontsize=9,
)

ax1.set_title("B Masked-region importance", fontsize=11, loc="left")
ax1.set_ylabel("ΔAUROC (baseline − masked)")
ax1.set_xlabel("Hotspot combination")

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# ============================================================
# C. Only performance（右下）
# ============================================================
ax2 = fig.add_subplot(gs[1, 1])

ax2.bar(df_plot["combo_label"], df_plot["AUC_only"])

ax2.axhline(baseline_auc, color="#2A6DB0", ls="--", lw=1)
ax2.text(
    len(order)-0.2,
    baseline_auc + 0.01,
    "baseline",
    fontsize=8,
    color=MAIN_COLOR
)

ax2.set_title("C Region-only performance", fontsize=11, loc="left")
ax2.set_ylabel("AUROC")
ax2.set_xlabel("Hotspot combination")

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# ============================================================
# 美化 x 轴
# ============================================================
for ax in [ax1, ax2]:
    ax.set_xticklabels(order, rotation=20)

plt.tight_layout()
plt.show()